# DZ17 — Нейронна мережа для розпізнавання чисел в інтервалі [5; 15]

У цьому завданні ми будуємо просту нейронну мережу на **PyTorch**, яка вчиться відрізняти числа:

- якщо число належить інтервалу від **5 до 15** включно — це **клас 1**;
- якщо число менше 5 або більше 15 — це **клас 0**.

Спочатку реалізуємо **базову модель** через `nn.Sequential`, а потім зробимо **ускладнену модель** у вигляді класу `Net`.
Після цього порівняємо навчання обох моделей та їхні лінії рішень.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Підготуємо навчальні дані
# Візьмемо числа на відрізку [0; 20] з рівномірним кроком
X = torch.linspace(0., 20., steps=100).unsqueeze(1)  # тензор розміру (100, 1)

# Створюємо мітки: 1, якщо число в [5; 15], інакше 0
y = ((X >= 5.) & (X <= 15.)).float()  # (100, 1)

X[:10], y[:10]

### Дослідження
- `torch.linspace(0., 20., steps=100)` генерує 100 чисел від 0 до 20.
- `unsqueeze(1)` додає стовпчиковий вимір, щоб форма була `(N, 1)` — так само, як у прикладі з днем тижня.
- Мітки `y` обчислюємо через логічну умову: якщо `x` у проміжку `[5; 15]`, значення `True`, в іншому випадку — `False`.
- Після `.float()` отримаємо 0.0 або 1.0, що підходить для `BCELoss`.

## Базова модель: `nn.Sequential`

Зробимо просту мережу через `Sequential`:
- прихований шар `Linear(1, 8)`;
- функція активації `ReLU`;
- вихідний шар `Linear(8, 1)`;
- `Sigmoid` на виході, щоб отримати ймовірність класу 1.

Такий варіант вже може сформувати «плато» для інтервалу `[5; 15]`, на відміну від моделі з одним нейроном.

In [ ]:
# Базова модель (Sequential)
model = nn.Sequential(
  nn.Linear(1, 8),  # 1 -> 8
  nn.ReLU(),        # нелінійність
  nn.Linear(8, 1),  # 8 -> 1
  nn.Sigmoid()      # ймовірність класу 1
)

loss_fn = nn.BCELoss()
optim = torch.optim.SGD(model.parameters(), lr=.1)

### Дослідження
- `nn.Linear(1, 8)` перетворює одне число у вектор з 8 ознак.
- `ReLU` додає нелінійність, щоб модель могла побудувати складнішу форму залежності.
- `nn.Linear(8, 1)` зводить 8 ознак до одного логіта.
- `Sigmoid` переводить логіт у число від 0 до 1 — це ймовірність того, що число належить інтервалу `[5; 15]`.
- `BCELoss` працює з ймовірностями і мітками 0/1.
- Оптимізатор `SGD` з відносно великою швидкістю навчання `0.1`.

In [ ]:
# Навчання базової моделі
n_epochs = 2000
loss_history_basic = []

for epoch in range(n_epochs):
  y_pred = model(X)
  loss = loss_fn(y_pred, y)

  optim.zero_grad()
  loss.backward()
  optim.step()

  loss_history_basic.append(loss.item())

  if epoch % 200 == 0:
    print(f"Epoch: {epoch} | Loss: {loss.item():.4f}")

loss_history_basic[-1]

### Дослідження
- Цикл навчання за схемою: `forward -> loss -> zero_grad -> backward -> step`.
  1. `zero_grad()` — обнуляємо градієнти;
  2. `loss.backward()` — обчислюємо градієнти;
  3. `optim.step()` — оновлюємо ваги.
- Зберігаємо `loss.item()` на кожній епосі у `loss_history_basic`, щоб потім подивитися графік навчання.
- Останнє значення loss має бути вже невеликим — це означає, що модель навчилася добре відокремлювати інтервал.

In [ ]:
# Графік зміни помилки під час навчання базової моделі
plt.plot(loss_history_basic)
plt.xlabel('Епоха')
plt.ylabel('Loss')
plt.title('Навчання базової моделі')
plt.show()

### Дослідження
- Крива помилки показує, як модель поступово навчається.
- Якщо графік спадає і виходить на плато, це хороша ознака.
- Якби помилка росла або хаотично стрибала, треба було б змінити архітектуру або швидкість навчання.

In [ ]:
# Тестування базової моделі на вигаданих даних
test = torch.tensor([[0.], [3.], [5.], [7.], [10.], [15.], [17.], [20.]], dtype=torch.float32)

with torch.no_grad():
  pred = model(test)

print("Тестові числа:", test.squeeze().tolist())
print("Ймовірності класу 1 (basic):", [round(p.item(), 3) for p in pred])
print("Класи (>= 0.5 -> 1):", [int(p.item() >= 0.5) for p in pred])

### Дослідження
- Перевіряємо модель на конкретних числах: `0, 3, 5, 7, 10, 15, 17, 20`.
- Усередині інтервалу `[5; 15]` очікуємо ймовірності ближчі до 1, поза інтервалом — ближчі до 0.
- Поріг `0.5` перетворює ймовірності у класи 0/1.

In [ ]:
# Візуалізація лінії рішення базової моделі
with torch.no_grad():
  grid_x = torch.linspace(0., 20., steps=400).unsqueeze(1)
  grid_proba_basic = model(grid_x).squeeze(1)

plt.figure(figsize=(7, 4))
plt.plot(grid_x.numpy(), grid_proba_basic.numpy())
plt.axvspan(5, 15, color='green', alpha=.1, label='Інтервал [5; 15]')
plt.axhline(0.5, color='red', linestyle='--', label='Поріг 0.5')
plt.xlabel('x')
plt.ylabel('P(клас = 1)')
plt.title('Лінія рішення базової моделі')
plt.legend()
plt.show()

### Дослідження
- По осі `x` — числа від 0 до 20, по осі `y` — ймовірність того, що число в інтервалі `[5; 15]`.
- Зелена зона показує справжній інтервал `5–15`.
- Лінія моделі має бути високою (близько 1) всередині зеленої зони та низькою (близько 0) поза нею.

## Ускладнена модель: клас `Net` з кількома шарами

Тепер зробимо мережу складнішою:
- оголосимо власний клас `Net`, який наслідується від `nn.Module`;
- додамо ще один прихований шар;
- використаємо різні функції активації (`Tanh`, `ReLU`, `Sigmoid`).

In [ ]:
# Оголошення складнішої мережі
class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1 = nn.Linear(1, 8)
    self.layer2 = nn.Linear(8, 8)
    self.layer3 = nn.Linear(8, 1)
    self.tanh = nn.Tanh()
    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()

  def forward(self, X):
    X = self.layer1(X)
    X = self.tanh(X)
    X = self.layer2(X)
    X = self.relu(X)
    X = self.layer3(X)
    X = self.sigmoid(X)
    return X

model2 = Net()
loss_fn2 = nn.BCELoss()
optim2 = torch.optim.Adam(model2.parameters(), lr=.01)

### Дослідження
- Архітектура тепер трохи складніша:
  - `Linear(1 → 8)` + `Tanh` — перший прихований шар (як у NLP-прикладах);
  - `Linear(8 → 8)` + `ReLU` — другий прихований шар (як у задачах Computer Vision);
  - `Linear(8 → 1)` + `Sigmoid` — вихідний шар з ймовірністю класу.
- У `Net` є три лінійні шари: `1 -> 8 -> 8 -> 1`.
- Після першого шару використовуємо `Tanh`, після другого — `ReLU`.
- `Sigmoid` на виході знову дає ймовірність класу 1.
- Оптимізатор `Adam` автоматично підлаштовує крок навчання для кожного параметра, тому часто сходиться швидше, ніж `SGD`.
- Така модель має більше параметрів і може вчити складніші закономірності.

In [ ]:
# Навчання складнішої моделі
n_epochs2 = 2000
loss_history_deep = []

for epoch in range(n_epochs2):
  y_pred2 = model2(X)
  loss2 = loss_fn2(y_pred2, y)

  optim2.zero_grad()
  loss2.backward()
  optim2.step()

  loss_history_deep.append(loss2.item())

  if epoch % 200 == 0:
    print(f"[Net] Epoch: {epoch} | Loss: {loss2.item():.4f}")

loss_history_deep[-1]

### Дослідження
- Цикл навчання такий самий, як і для базової моделі, але тепер оновлюємо параметри `model2`.
- Знову зберігаємо помилку у `loss_history_deep`, щоб можна було порівняти криві навчання двох моделей.

In [ ]:
# Графік навчання ускладненої моделі
plt.plot(loss_history_deep)
plt.xlabel('Епоха')
plt.ylabel('Loss')
plt.title('Навчання ускладненої моделі (Net)')
plt.show()

# Порівняння кривих навчання двох моделей на одному графіку
plt.plot(loss_history_basic, label='Базова модель (Sequential + SGD)')
plt.plot(loss_history_deep, label='Ускладнена модель (Net + Adam)')
plt.xlabel('Епоха')
plt.ylabel('Loss')
plt.title('Порівняння навчання моделей')
plt.legend()
plt.show()

### Дослідження
- Перший графік показує, як навчається тільки `Net`.
- Другий графік дає можливість візуально порівняти, яка модель швидше і нижче зменшує помилку.
- Якщо крива `Net` падає швидше або до нижчого рівня — це означає, що ускладнена архітектура + Adam працюють краще.

In [ ]:
# Тестування ускладненої моделі на тих самих даних
with torch.no_grad():
  pred2 = model2(test)

print("Тестові числа:", test.squeeze().tolist())
print("Ймовірності класу 1 (Net):", [round(p.item(), 3) for p in pred2])
print("Класи (>= 0.5 -> 1):", [int(p.item() >= 0.5) for p in pred2])

### Дослідження
- Використовуємо ті самі тестові числа, щоб чесно порівняти роботу моделей.
- Дивимось, чи відрізняються ймовірності: інколи `Net` дає більш «рішучі» значення, ближчі до 0 або 1.

In [ ]:
# Візуалізація лінії рішення ускладненої моделі
with torch.no_grad():
  grid_x = torch.linspace(0., 20., steps=400).unsqueeze(1)
  grid_proba_deep = model2(grid_x).squeeze(1)

plt.figure(figsize=(7, 4))
plt.plot(grid_x.numpy(), grid_proba_deep.numpy())
plt.axvspan(5, 15, color='green', alpha=.1, label='Інтервал [5; 15]')
plt.axhline(0.5, color='red', linestyle='--', label='Поріг 0.5')
plt.xlabel('x')
plt.ylabel('P(клас = 1)')
plt.title('Лінія рішення ускладненої моделі (Net)')
plt.legend()
plt.show()

### Дослідження
- Тут бачимо, як ускладнена модель розділяє числа на всьому відрізку `[0; 20]`.
- Ідеально, якщо ймовірність близька до 1 в `[5; 15]` та близька до 0 поза цим інтервалом.
- Якщо форма кривої відрізняється від базової моделі, можна обговорити, яка з них краще відображає інтервал.

## Підсумок

- **Базова модель** на `Sequential` з одним прихованим шаром уже добре вчиться розрізняти числа в інтервалі `[5; 15]`.
- **Ускладнена модель** `Net` з додатковим шаром і активаціями `Tanh + ReLU` демонструє, як можна об'єднати ідеї з попередніх прикладів, які ми пройшли.
- Графіки навчання та ліній рішень дозволяють візуально порівняти, як кожна модель поводиться на даних.
- Важливо не лише отримати низький loss, але й подивитися на **лінію рішення** та якість передбачень на тестових прикладах.
- Такий підхід відповідає "науковому міні-дослідженню": ми пробуємо різні архітектури та аналізуємо, як вони вчаться визначати числа від 5 до 15.